In [ ]:
import gradio as gr
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

# --- 1. לוגיקת החיזוי של המניות ---
def predict_logic(symbol):
    try:
        # הורדת נתונים (חודש אחרון)
        df = yf.download(symbol, period="1mo", interval="1d")

        if df.empty:
            return "לא נמצאו נתונים עבור הסימול הזה", None, None

        # ניקוי כותרות (Multi-index) - חשוב לגרסאות חדשות של yfinance
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
            if 'Ticker' in df.columns:
                df = df.drop(columns=['Ticker'])

        # בחירת נתונים אחרונים
        last_price = float(df['Close'].iloc[-1])
        prev_price = float(df['Close'].iloc[-2])

        # קביעת מגמה (לוגיקה בסיסית)
        diff = last_price - prev_price
        trend = f"עליה (UP) 🚀" if diff > 0 else f"ירידה (DOWN) 📉"
        change_pct = (diff / prev_price) * 100

        res_text = f"מגמה: {trend}\nמחיר אחרון: {last_price:.2f}\nשינוי יומי: {change_pct:.2f}%"

        # הסתברות (כרגע מבוסס על המגמה האחרונה)
        prob = {"עליה": 0.65 if diff > 0 else 0.35, "ירידה": 0.35 if diff > 0 else 0.65}

        # יצירת גרף
        fig, ax = plt.subplots(figsize=(8, 4))
        # צביעת הגרף לפי המגמה
        graph_color = 'green' if diff > 0 else 'red'
        df['Close'].plot(ax=ax, color=graph_color, marker='o')

        ax.set_title(f"Price History: {symbol}")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()

        return res_text, prob, fig

    except Exception as e:
        return f"שגיאה: {str(e)}", None, None

# --- 2. בניית הממשק ב-Gradio ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 חוזה מגמות בורסה")
    gr.Markdown("אפליקציה זו רצה באופן קבוע על שרתי Hugging Face.")

    with gr.Row():
        input_symbol = gr.Textbox(label="הכנס סימול (למשל AAPL, TSLA, ^GSPC)", value="^GSPC")
        btn = gr.Button("נתח עכשיו", variant="primary")

    with gr.Row():
        with gr.Column():
            res_text = gr.Textbox(label="תוצאה")
            res_label = gr.Label(label="תחזית מודל (הערכה)")
        with gr.Column():
            res_plot = gr.Plot(label="גרף מחירים")

    btn.click(fn=predict_logic, inputs=input_symbol, outputs=[res_text, res_label, res_plot])

# --- 3. הרצה ---
if __name__ == "__main__":
    # ב-Hugging Face אין צורך להגדיר פורט או share=True
    demo.launch()